In [1]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "brown"

In [4]:
from langchain.agents import create_agent
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.8,
)

agent = create_agent(
    model=model,
    context_schema=ColourContext  
)

/mnt/c/Users/wstre/projects/lca-lc-foundations/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [6]:
from pprint import pprint

pprint(response)
print(response["messages"][-1].content)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='d2a3b2ab-7a9e-417b-98ab-f30a685d11dc'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "The image provided does not contain any visible content, so it's not possible to determine a favorite color.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The image provided does not contain any visible content, so it's not possible to determine a favorite color."}]}, response_metadata={'model_name': 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning-20260428:free', 'id': 'gen-1779048621-tD12kUQN0Pd5YSoFaNwl', 'created': 1779048621, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019e378f-a2a8-7983-93b3-376727aec372-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 22, 'total_tokens': 44, 'input_token_de

## Accessing Context

In [7]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [8]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

/mnt/c/Users/wstre/projects/lca-lc-foundations/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...avourite_colour='brown'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='4daa783a-d461-45a7-b94b-cba3b7def816'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking, "What is my favourite colour?" Let me see how to approach this.\n\nFirst, I need to check if there are any tools available that can help answer this. The tools provided are get_favourite_colour and get_least_favourite_colour. Both functions don\'t require any parameters, so they should retrieve the respective colours based on the user\'s data.\n\nSince the user is asking for their favourite colour, the relevant function here is get_favourite_colour. The other function is for the least favourite, which isn\'t needed here. \n\nI should call get_favourite_colour. There are no parameters required, so the tool call is straightforward. No need to ask the user for more information since the function should handle retrieving the data. \n\nJ

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)
print(response["messages"][-1].content)

/mnt/c/Users/wstre/projects/lca-lc-foundations/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...avourite_colour='brown'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='770ce00e-377c-48cc-ae75-b9e18f6ae9f2'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for their favorite color. Let me check the tools available. There's a function called get_favourite_colour with no parameters. Since the user wants their favorite color, I should call that function. No need for any arguments here. Just trigger the function and return the result.\n", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "Okay, the user is asking for their favorite color. Let me check the tools available. There's a function called get_favourite_colour with no parameters. Since the user wants their favorite color, I should call that function. No need for any arguments here. Just trigger the function and return the result.\n"}]}, response_metadata={'model_name': 'nvidia/nemotron-3-nano